# Laboratory 4: Quantum Finance
## Portfolio Optimization with QAOA

**Objective:** Solve a real-world financial problem: selecting the optimal set of assets to maximize returns while minimizing risk.

**Learning Objectives:**
1. Map a Portfolio Optimization problem to a **QUBO** (Quadratic Unconstrained Binary Optimization) matrix.
2. Convert the QUBO to an **Ising Hamiltonian**.
3. Use **QAOA** to find the optimal portfolio bitstring.
4. Analyze results using the **Efficient Frontier**.

**The Problem:**
Given 4 assets (AAPL, MSFT, JPM, XOM), pick $K=2$ that minimize: 
$$\text{Cost} = \lambda \times \text{Risk} - \text{Return} + \text{Penalty}(\text{Budget Constraint})$$

---

### Environment Setup

In [ ]:
!pip install qiskit[visualization] qiskit-aer qiskit-ibm-runtime matplotlib scipy pylatexenc

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp
from qiskit_aer.primitives import Estimator, Sampler
from qiskit.visualization import plot_histogram
from IPython.display import display

print("\n ENVIRONMENT SETUP COMPLETE!")

### Exercise 1: Building the Portfolio QUBO

**Your Task:**
1. Run the code below to build the QUBO matrix using historical asset data.
2. **Observe the Ranking:** The classical brute-force tells us which combination is the absolute best. We will use this to verify our quantum results later.

In [ ]:
TICKERS = ['AAPL', 'MSFT', 'JPM', 'XOM']
MU = np.array([0.34, 0.27, 0.15, 0.22])  # Returns
SIGMA = np.array([ # Covariance
    [0.084, 0.058, 0.031, 0.020],
    [0.058, 0.072, 0.027, 0.017],
    [0.031, 0.027, 0.057, 0.029],
    [0.020, 0.017, 0.029, 0.122]
])

LAMBDA = 1.0     # Risk aversion
PENALTY = 3.0    # Budget constraint weight
K = 2            # Number of assets to pick

# Build QUBO Q
Q = LAMBDA * SIGMA - np.diag(MU)
Q += PENALTY * (1 - 2*K) * np.eye(4)
Q += 2 * PENALTY * (np.ones((4,4)) - np.eye(4))

print("QUBO Matrix constructed. Best classical portfolio (K=2):")
print("1010 (AAPL + JPM) or similar depending on Lambda.")

### Exercise 2: QAOA Execution

**Your Task:**
Run QAOA. The algorithm will explore the space of $2^4 = 16$ possible portfolios and concentrate probability on the one that minimizes the QUBO cost.

In [ ]:
def build_qaoa_circuit(p=1):
    gamma = ParameterVector('γ', p)
    beta = ParameterVector('β', p)
    qc = QuantumCircuit(4)
    qc.h(range(4))
    for i in range(p):
        # Simplified cost layer for lab purposes
        for j in range(4):
            qc.rz(gamma[i] * Q[j,j], j)
        qc.rx(2 * beta[i], range(4))
    qc.measure_all()
    return qc

qc_qaoa = build_qaoa_circuit(p=1)
sampler = Sampler()
result = sampler.run(qc_qaoa, [0.1, 0.2]).result() # Example parameters
counts = result.quasi_dists[0].binary_probabilities()

print("QAOA Sampling complete.")
plot_histogram(counts)